# TP Argumentation 

## But du TP

> On veut créer un programme qui lit un graphe d'argumentation et calcule les arguments acceptables selon :

- Complete semantics
- Grounded semantics
- Preferred semantics

Exemple de graphe :

```prolog
arg(a).
arg(b).
arg(c).
att(a,b).
att(b,c).
```

Cela veut dire :

```text
a → b → c
```


# 1. Importation des bibliothèques

- `combinations` sert à générer tous les sous-ensembles.
- `re` sert à reconnaître les lignes `arg(a).` et `att(a,b).`.


In [6]:
from itertools import combinations
import re

# 2. Lecture du graphe

Cette fonction transforme le texte du graphe en deux listes :

```python
arguments = ["a", "b", "c"]
attacks = [("a", "b"), ("b", "c")]
```


In [7]:
def parse_argumentation_framework(text):
    """
    Lit un graphe d'argumentation écrit avec :
    arg(a).
    att(a,b).

    Retourne :
    - arguments : liste des arguments
    - attacks : liste des attaques
    """

    # Liste des arguments du graphe.
    arguments = []

    # Liste des attaques sous forme de couples (attaquant, attaqué).
    attacks = []

    # Découper le texte ligne par ligne.
    lines = text.strip().splitlines()

    # Lire chaque ligne.
    for line in lines:

        # Enlever les espaces inutiles.
        line = line.strip().replace(" ", "")

        # Ignorer les lignes vides ou les commentaires.
        if not line or line.startswith("#"):
            continue

        # Vérifier si la ligne ressemble à arg(a).
        arg_match = re.fullmatch(r"arg\(([^()]+)\)\.", line)

        # Vérifier si la ligne ressemble à att(a,b).
        att_match = re.fullmatch(r"att\(([^(),]+),([^(),]+)\)\.", line)

        # Cas 1 : la ligne est un argument.
        if arg_match:
            arg = arg_match.group(1)

            # Ajouter l'argument s'il n'existe pas déjà.
            if arg not in arguments:
                arguments.append(arg)

        # Cas 2 : la ligne est une attaque.
        elif att_match:
            attacker = att_match.group(1)
            attacked = att_match.group(2)

            # Ajouter l'attaque.
            attacks.append((attacker, attacked))

            # Sécurité : ajouter automatiquement les arguments oubliés.
            if attacker not in arguments:
                arguments.append(attacker)
            if attacked not in arguments:
                arguments.append(attacked)

        # Cas 3 : ligne mal écrite.
        else:
            raise ValueError(f"Ligne non reconnue : {line}")

    return sorted(arguments), attacks

## Test de lecture du graphe

In [8]:
exemple_graphe = """
arg(a).
arg(b).
arg(c).
att(a,b).
att(b,c).
"""

arguments, attacks = parse_argumentation_framework(exemple_graphe)

print("Arguments :", arguments)
print("Attaques  :", attacks)

Arguments : ['a', 'b', 'c']
Attaques  : [('a', 'b'), ('b', 'c')]


# 3. Générer tous les sous-ensembles

Pour trouver les extensions, on doit tester toutes les combinaisons possibles.

Avec `{a,b,c}`, on obtient :

```text
{}
{a}
{b}
{c}
{a,b}
{a,c}
{b,c}
{a,b,c}
```

Avec 3 arguments, cela fait `2^3 = 8` sous-ensembles.


In [9]:
def powerset(arguments):
    """
    Génère tous les sous-ensembles possibles.
    """

    return [
        set(comb)
        for r in range(len(arguments) + 1)
        for comb in combinations(arguments, r)
    ]

## Test des sous-ensembles

In [10]:
for S in powerset(["a", "b", "c"]):
    print(S)

set()
{'a'}
{'b'}
{'c'}
{'a', 'b'}
{'c', 'a'}
{'c', 'b'}
{'c', 'a', 'b'}


# 4. Vérifier si un ensemble est conflict-free

Un ensemble est conflict-free si aucun argument de cet ensemble n'attaque un autre argument du même ensemble.

Exemple :

```text
a → b
```

- `{a,b}` n'est pas conflict-free.
- `{a}` est conflict-free.
- `{b}` est conflict-free.


In [11]:
def is_conflict_free(S, attacks):
    """
    Vérifie si S est conflict-free.
    """

    for attacker, attacked in attacks:

        # Si l'attaquant et l'attaqué sont tous les deux dans S,
        # alors il y a un conflit.
        if attacker in S and attacked in S:
            return False

    return True

## Test conflict-free

In [12]:
print("{a,b} conflict-free ?", is_conflict_free({"a", "b"}, attacks))
print("{a,c} conflict-free ?", is_conflict_free({"a", "c"}, attacks))

{a,b} conflict-free ? False
{a,c} conflict-free ? True


# 5. Trouver les attaquants d'un argument

Cette fonction permet de savoir qui attaque un argument donné.

Exemple :

```text
a → b ← c
```

Les attaquants de `b` sont `{a,c}`.


In [13]:
def attackers_of(argument, attacks):
    """
    Retourne tous les arguments qui attaquent l'argument donné.
    """

    return {
        attacker
        for attacker, attacked in attacks
        if attacked == argument
    }

## Test des attaquants

In [14]:
print("Attaquants de b :", attackers_of("b", attacks))
print("Attaquants de c :", attackers_of("c", attacks))

Attaquants de b : {'a'}
Attaquants de c : {'b'}


# 6. Vérifier si un ensemble défend un argument

Un ensemble `S` défend un argument si tous les attaquants de cet argument sont attaqués par au moins un élément de `S`.

Exemple :

```text
a → b → c
```

`b` attaque `c`, mais `a` attaque `b`, donc `{a}` défend `c`.


In [15]:
def defends(S, argument, attacks):
    """
    Vérifie si l'ensemble S défend un argument.
    """

    attackers = attackers_of(argument, attacks)

    for attacker in attackers:
        counter_attacked = False

        # Chercher si un élément de S attaque l'attaquant.
        for x, y in attacks:
            if x in S and y == attacker:
                counter_attacked = True
                break

        # Si un attaquant n'est pas neutralisé, l'argument n'est pas défendu.
        if not counter_attacked:
            return False

    return True

## Test de défense

In [16]:
print("{a} défend c ?", defends({"a"}, "c", attacks))
print("{c} défend c ?", defends({"c"}, "c", attacks))

{a} défend c ? True
{c} défend c ? False


# 7. Vérifier si un ensemble est admissible

Un ensemble est admissible si :

1. il est conflict-free ;
2. il défend tous ses arguments.

Donc :

```text
admissible = sans conflit + défendu
```


In [17]:
def is_admissible(S, attacks):
    """
    Vérifie si un ensemble S est admissible.
    """

    # Condition 1 : S doit être sans conflit.
    if not is_conflict_free(S, attacks):
        return False

    # Condition 2 : chaque argument de S doit être défendu par S.
    for argument in S:
        if not defends(S, argument, attacks):
            return False

    return True

## Test admissible

In [18]:
print("{a,c} admissible ?", is_admissible({"a", "c"}, attacks))
print("{b} admissible ?", is_admissible({"b"}, attacks))

{a,c} admissible ? True
{b} admissible ? False


# 8. Calcul des extensions complètes

Une extension complète est :

- admissible ;
- contient tous les arguments qu'elle défend.

Exemple avec :

```text
a → b → c
```

`{a}` défend aussi `c`, donc `{a}` n'est pas complète.  
`{a,c}` est complète.


In [19]:
def complete_extensions(arguments, attacks):
    """
    Calcule toutes les extensions complètes.
    """

    complete_exts = []

    # Tester tous les sous-ensembles.
    for S in powerset(arguments):

        # Garder seulement les ensembles admissibles.
        if is_admissible(S, attacks):

            # Trouver tous les arguments défendus par S.
            defended_arguments = {
                arg
                for arg in arguments
                if defends(S, arg, attacks)
            }

            # S doit contenir tous les arguments qu'il défend.
            if defended_arguments.issubset(S):
                complete_exts.append(S)

    return complete_exts

## Test complete semantics

In [20]:
co = complete_extensions(arguments, attacks)

print("Extensions complètes :")
for ext in co:
    print(ext)

Extensions complètes :
{'c', 'a'}


# 9. Calcul de l'extension grounded

L'extension grounded est la plus petite extension complète par inclusion.

Elle représente la solution la plus prudente.


In [21]:
def grounded_extension(arguments, attacks):
    """
    Calcule l'extension grounded.
    """

    complete_exts = complete_extensions(arguments, attacks)

    # Chercher l'extension minimale.
    for S in complete_exts:
        is_minimal = True

        for T in complete_exts:
            if T < S:
                is_minimal = False
                break

        if is_minimal:
            return S

    return set()

## Test grounded semantics

In [22]:
gr = grounded_extension(arguments, attacks)
print("Extension grounded :", gr)

Extension grounded : {'c', 'a'}


# 10. Calcul des extensions préférées

Les extensions préférées sont les plus grandes extensions complètes par inclusion.


In [23]:
def preferred_extensions(arguments, attacks):
    """
    Calcule les extensions préférées.
    """

    complete_exts = complete_extensions(arguments, attacks)
    preferred_exts = []

    # Chercher les extensions maximales.
    for S in complete_exts:
        is_maximal = True

        for T in complete_exts:
            if S < T:
                is_maximal = False
                break

        if is_maximal:
            preferred_exts.append(S)

    return preferred_exts

## Test preferred semantics

In [24]:
pr = preferred_extensions(arguments, attacks)

print("Extensions préférées :")
for ext in pr:
    print(ext)

Extensions préférées :
{'c', 'a'}


# 11. Fonction finale

Cette fonction regroupe tout :

1. lire le graphe ;
2. calculer les extensions complètes ;
3. calculer grounded ;
4. calculer preferred ;
5. afficher les résultats.


In [25]:
def format_extension(S):
    """
    Affiche une extension proprement.
    """

    if not S:
        return "∅"

    return "{" + ", ".join(sorted(S)) + "}"


def analyze_af(text):
    """
    Analyse complète d'un graphe d'argumentation.
    """

    arguments, attacks = parse_argumentation_framework(text)

    co = complete_extensions(arguments, attacks)
    gr = grounded_extension(arguments, attacks)
    pr = preferred_extensions(arguments, attacks)

    print("===== Graphe d'argumentation =====")
    print("Arguments :", arguments)
    print("Attaques  :", attacks)

    print("\n===== Résultats =====")

    print("\nComplete semantics:")
    for ext in co:
        print("  ", format_extension(ext))

    print("\nGrounded semantics:")
    print("  ", format_extension(gr))

    print("\nPreferred semantics:")
    for ext in pr:
        print("  ", format_extension(ext))

    return {
        "arguments": arguments,
        "attacks": attacks,
        "complete": co,
        "grounded": gr,
        "preferred": pr
    }

# 12. Exemple 1 — Graphe sans cycle

```text
a → b → c
```

Résultat attendu :

```text
{a,c}
```


In [26]:
graph1 = """
arg(a).
arg(b).
arg(c).
att(a,b).
att(b,c).
"""

result1 = analyze_af(graph1)

===== Graphe d'argumentation =====
Arguments : ['a', 'b', 'c']
Attaques  : [('a', 'b'), ('b', 'c')]

===== Résultats =====

Complete semantics:
   {a, c}

Grounded semantics:
   {a, c}

Preferred semantics:
   {a, c}


# 13. Exemple 2 — Cycle pair

```text
a ↔ b
```

On ne peut pas accepter `a` et `b` ensemble.

Grounded est vide, car aucune solution n'est plus sûre que l'autre.


In [27]:
graph2 = """
arg(a).
arg(b).
att(a,b).
att(b,a).
"""

result2 = analyze_af(graph2)

===== Graphe d'argumentation =====
Arguments : ['a', 'b']
Attaques  : [('a', 'b'), ('b', 'a')]

===== Résultats =====

Complete semantics:
   ∅
   {a}
   {b}

Grounded semantics:
   ∅

Preferred semantics:
   {a}
   {b}


# 14. Exemple 3 — Cycle impair

```text
a → b → c → a
```

Chaque argument est attaqué par un autre.  
Grounded est souvent vide dans ce cas.


In [28]:
graph3 = """
arg(a).
arg(b).
arg(c).
att(a,b).
att(b,c).
att(c,a).
"""

result3 = analyze_af(graph3)

===== Graphe d'argumentation =====
Arguments : ['a', 'b', 'c']
Attaques  : [('a', 'b'), ('b', 'c'), ('c', 'a')]

===== Résultats =====

Complete semantics:
   ∅

Grounded semantics:
   ∅

Preferred semantics:
   ∅


# 15. Graphe à modifier

Tu peux changer ce graphe pour tester d'autres exemples.


In [29]:
mon_graphe = """
arg(a).
arg(b).
arg(c).
arg(d).

att(a,b).
att(b,c).
att(c,b).
att(c,d).
"""

result_custom = analyze_af(mon_graphe)

===== Graphe d'argumentation =====
Arguments : ['a', 'b', 'c', 'd']
Attaques  : [('a', 'b'), ('b', 'c'), ('c', 'b'), ('c', 'd')]

===== Résultats =====

Complete semantics:
   {a, c}

Grounded semantics:
   {a, c}

Preferred semantics:
   {a, c}


# 16. Conclusion

Ce TP permet de comprendre comment calculer automatiquement les arguments acceptables dans un graphe d'argumentation.



> Mon algorithme teste toutes les combinaisons possibles d'arguments, puis il garde celles qui sont sans conflit et bien défendues. Ensuite, il calcule les extensions complètes, grounded et préférées.
